# EDA météo — Fourcasters

J'explore ici les données météo agrégées par **département et par jour**.

Le but est simple :
- vérifier que les données sont propres ;
- regarder les grandes tendances ;
- comparer les saisons et les territoires ;
- repérer les situations météo les plus marquées.

Pour avoir les librairies du notebook :
`uv sync --group analyse`

## 1. Chargement des données

Je pars de la table `int_meteo_departement_jour`. Elle est déjà agrégée au bon niveau pour l'analyse et évite de charger toutes les lignes des 360 POI.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from google.cloud import bigquery

from fourcasters_dbt.configuration import (
    PROJET_GCP,
    DATASET_ANALYSE,
    configurer_google_cloud,
)

configurer_google_cloud()
client = bigquery.Client(project=PROJET_GCP)

table_id = f"{PROJET_GCP}.{DATASET_ANALYSE}.int_meteo_departement_jour"
df = client.list_rows(table_id).to_dataframe(create_bqstorage_client=False)

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["date", "numero_departement"]).reset_index(drop=True)

df.head()

**Lecture :** une ligne correspond à un département pour une journée. C'est suffisant pour une EDA générale et beaucoup plus léger que les données POI brutes.

## 2. Vue générale

In [ ]:
print(f"Nombre de lignes : {len(df):,}")
print(f"Période : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Nombre de départements : {df['numero_departement'].nunique()}")
print(f"Nombre de jours : {df['date'].nunique()}")

display(df.dtypes.to_frame("type"))
display(df.head())

**Interprétation :** je vérifie surtout que la période commence bien autour de 2000 et que la couverture géographique correspond au projet.

## 3. Qualité des données

In [ ]:
qualite = pd.DataFrame({
    "valeurs_manquantes": df.isna().sum(),
    "pourcentage": (df.isna().mean() * 100).round(2),
}).sort_values("pourcentage", ascending=False)

doublons = df.duplicated(subset=["date", "numero_departement"]).sum()

print(f"Doublons département / jour : {doublons}")
display(qualite.head(15))

nb_departements = df["numero_departement"].nunique()

couverture = (
    df.groupby("date")
      .agg(
          departements=("numero_departement", "nunique"),
          departements_complets=("mesures_completes", "sum"),
      )
)

jours_incomplets = couverture[couverture["departements"] != nb_departements]

print(f"Jours avec une couverture incomplète : {len(jours_incomplets)}")
display(jours_incomplets.tail(10))

**Interprétation :** s'il n'y a pas de doublons et très peu de valeurs manquantes, les données sont suffisamment propres pour continuer. Les journées incomplètes sont à garder en tête dans les comparaisons.

## 4. Statistiques générales

In [ ]:
variables = [
    "temperature_moyenne",
    "temperature_minimale",
    "temperature_maximale",
    "humidite_moyenne",
    "precipitations_moyennes",
    "vitesse_vent_moyenne",
    "rafale_vent_maximale",
    "deficit_pression_vapeur_maximal",
]

statistiques = df[variables].describe().T.round(2)
display(statistiques)

**Interprétation :** ce tableau donne les ordres de grandeur et permet de repérer rapidement une valeur extrême ou incohérente.

## 5. Saisonnalité

Je regarde les moyennes par mois. Pour le projet incendie, c'est important car les conditions météo changent fortement entre l'hiver et l'été.

In [ ]:
df["mois"] = df["date"].dt.month

par_mois = (
    df.groupby("mois")[[
        "temperature_moyenne",
        "humidite_moyenne",
        "precipitations_moyennes",
        "rafale_vent_maximale",
        "deficit_pression_vapeur_maximal",
    ]]
    .mean()
    .round(2)
)

display(par_mois)

par_mois["temperature_moyenne"].plot(marker="o", figsize=(9, 4))
plt.title("Température moyenne selon le mois")
plt.xlabel("Mois")
plt.ylabel("Température (°C)")
plt.xticks(range(1, 13))
plt.show()

par_mois["humidite_moyenne"].plot(marker="o", figsize=(9, 4))
plt.title("Humidité moyenne selon le mois")
plt.xlabel("Mois")
plt.ylabel("Humidité (%)")
plt.xticks(range(1, 13))
plt.show()

par_mois["precipitations_moyennes"].plot(marker="o", figsize=(9, 4))
plt.title("Précipitations moyennes selon le mois")
plt.xlabel("Mois")
plt.ylabel("Précipitations (mm)")
plt.xticks(range(1, 13))
plt.show()

par_mois["deficit_pression_vapeur_maximal"].plot(marker="o", figsize=(9, 4))
plt.title("VPD maximal moyen selon le mois")
plt.xlabel("Mois")
plt.ylabel("VPD")
plt.xticks(range(1, 13))
plt.show()

**Interprétation :** on doit retrouver une saisonnalité nette. Les mois chauds sont particulièrement intéressants pour la suite car chaleur, air sec et manque de pluie peuvent favoriser un danger plus élevé.

## 6. Évolution par année

In [ ]:
df["annee"] = df["date"].dt.year

par_annee = (
    df.groupby("annee")
      .agg(
          jours=("date", "nunique"),
          temperature_moyenne=("temperature_moyenne", "mean"),
          precipitations_moyennes=("precipitations_moyennes", "mean"),
      )
      .round(2)
)

display(par_annee.tail())

annees_completes = par_annee[par_annee["jours"] >= 350]

annees_completes["temperature_moyenne"].plot(figsize=(10, 4), marker="o")
plt.title("Température moyenne par année")
plt.xlabel("Année")
plt.ylabel("Température moyenne (°C)")
plt.show()

annees_completes["precipitations_moyennes"].plot(figsize=(10, 4), marker="o")
plt.title("Précipitations moyennes par année")
plt.xlabel("Année")
plt.ylabel("Précipitations moyennes (mm)")
plt.show()

**Interprétation :** je garde seulement les années presque complètes pour éviter de comparer une année entière avec quelques mois seulement. L'idée est de voir la tendance générale, pas d'expliquer chaque variation annuelle.

## 7. Comparaison des départements pendant la saison des feux

Je me concentre sur **juin à septembre**, période la plus intéressante pour le projet.

In [ ]:
saison_feux = df[df["mois"].between(6, 9)].copy()

par_departement = (
    saison_feux.groupby(["numero_departement", "departement"])
    .agg(
        temperature_max=("temperature_maximale", "mean"),
        humidite=("humidite_moyenne", "mean"),
        precipitations=("precipitations_moyennes", "mean"),
        rafale=("rafale_vent_maximale", "mean"),
        vpd=("deficit_pression_vapeur_maximal", "mean"),
    )
    .round(2)
    .reset_index()
)

print("Départements avec le VPD moyen le plus élevé")
display(par_departement.sort_values("vpd", ascending=False).head(10))

print("Départements avec l'humidité moyenne la plus faible")
display(par_departement.sort_values("humidite").head(10))

top_vpd = par_departement.nlargest(10, "vpd").sort_values("vpd")
top_vpd.plot(
    x="departement",
    y="vpd",
    kind="barh",
    figsize=(9, 5),
    legend=False,
)
plt.title("VPD moyen le plus élevé de juin à septembre")
plt.xlabel("VPD moyen")
plt.ylabel("")
plt.show()

**Interprétation :** les conditions estivales ne sont pas les mêmes partout. Certains départements cumulent plus souvent chaleur et air sec, ce qui les rend intéressants à comparer avec le danger Météo-France.

## 8. Relations entre les variables météo

In [ ]:
correlations = df[[
    "temperature_moyenne",
    "temperature_maximale",
    "humidite_moyenne",
    "precipitations_moyennes",
    "rafale_vent_maximale",
    "deficit_pression_vapeur_maximal",
]].corr().round(2)

display(correlations)

fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(correlations, vmin=-1, vmax=1)

ax.set_xticks(range(len(correlations.columns)))
ax.set_xticklabels(correlations.columns, rotation=45, ha="right")
ax.set_yticks(range(len(correlations.index)))
ax.set_yticklabels(correlations.index)

plt.colorbar(image, ax=ax)
plt.title("Corrélations entre les variables météo")
plt.tight_layout()
plt.show()

**Interprétation :** certaines variables évoluent ensemble, notamment la température et le VPD, alors que l'humidité peut évoluer dans le sens inverse. Une corrélation ne veut pas dire qu'une variable cause l'autre.

## 9. Journées météo les plus marquées

In [ ]:
colonnes = [
    "date",
    "numero_departement",
    "departement",
    "temperature_maximale",
    "humidite_moyenne",
    "precipitations_moyennes",
    "rafale_vent_maximale",
    "deficit_pression_vapeur_maximal",
]

print("VPD les plus élevés")
display(
    df.nlargest(10, "deficit_pression_vapeur_maximal")[colonnes]
)

print("Humidité la plus faible")
display(
    df.nsmallest(10, "humidite_moyenne")[colonnes]
)

print("Rafales les plus fortes")
display(
    df.nlargest(10, "rafale_vent_maximale")[colonnes]
)

**Interprétation :** les journées extrêmes sont intéressantes parce qu'un danger élevé peut venir d'une combinaison de chaleur, sécheresse de l'air et vent plutôt que d'une seule variable.

## 10. Lecture rapide

In [ ]:
noms_mois = {
    1: "janvier", 2: "février", 3: "mars", 4: "avril",
    5: "mai", 6: "juin", 7: "juillet", 8: "août",
    9: "septembre", 10: "octobre", 11: "novembre", 12: "décembre",
}

mois_plus_chaud = par_mois["temperature_moyenne"].idxmax()
mois_plus_sec = par_mois["humidite_moyenne"].idxmin()
mois_vpd_max = par_mois["deficit_pression_vapeur_maximal"].idxmax()

print("Quelques constats :")
print(f"- Mois le plus chaud en moyenne : {noms_mois[mois_plus_chaud]}")
print(f"- Mois avec l'humidité la plus faible : {noms_mois[mois_plus_sec]}")
print(f"- Mois avec le VPD moyen le plus élevé : {noms_mois[mois_vpd_max]}")
print(f"- Jours incomplets dans la couverture : {len(jours_incomplets)}")

## Bilan

L'EDA montre surtout trois choses :
- la qualité et la couverture des données peuvent être vérifiées simplement ;
- la météo française est très saisonnière et varie selon les départements ;
- température, humidité, vent et VPD sont des variables intéressantes à rapprocher du danger incendie.

La suite logique est donc de comparer ces conditions météo avec les niveaux de danger Météo-France.